# Notebook 2: Speculative inference and facet generation

This notebook shows the lightweight speculative pipeline: write a working item, generate claims, persist branch-local memories, rank inference candidates, and inspect facet relations.

In [ ]:
from __future__ import annotations

import json
import shutil
import sys
from pathlib import Path
from pprint import pprint


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src").exists():
            return candidate
    raise RuntimeError("Could not find the repository root.")


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))


def show(data):
    if isinstance(data, str):
        print(data)
    else:
        print(json.dumps(data, indent=2, default=str))


In [ ]:
from src.active_memory.models import WorkingItem
from src.manifold_sidecar import ManifoldRankingService
from src.persistence.pipeline import MutationPipeline
from src.retrieval.composer import RetrievalComposer
from src.terminus.adapter import TerminusMemoryRepository

memory_root = REPO_ROOT / "notebooks" / ".demo-memory-02"
shutil.rmtree(memory_root, ignore_errors=True)
# This intentionally points at an unused local address so the repository
# uses its in-process fallback store without requiring a live TerminusDB server.
repo = TerminusMemoryRepository(url="http://localhost:9999")
service = ManifoldRankingService(model_id="notebook-ranker-v1", geometry_version="demo-geometry-v1")
pipeline = MutationPipeline(
    memory_root,
    enable_terminus=True,
    terminus_repo=repo,
    manifold_service=service,
    enable_inference=True,
)
composer = RetrievalComposer(memory_root, terminus_repo=repo)
service.get_current_model_metadata()

In [ ]:
item = WorkingItem(
    item_type="observation",
    content=(
        "Yesterday, Acme Payments delayed refund processing for enterprise customers. "
        "Today, Acme Payments restored refund processing after the batch job was retried."
    ),
    session_id="facet-demo",
)
result = pipeline.run(item, session_id="facet-demo")
show(result.__dict__)

In [ ]:
inference_nodes = repo.query_inference_nodes(result.inference_branch)
facet_relations = repo.query_facet_relations(result.inference_branch)
show({
    "inference_branch": result.inference_branch,
    "inference_nodes": inference_nodes,
    "facet_relations": facet_relations,
})

In [ ]:
trusted_context = composer.retrieve(include_terminus=True)
exploratory_context = composer.retrieve(
    include_terminus=True,
    include_speculative=True,
    inference_branch=result.inference_branch,
)
show({
    "trusted_speculative_results": trusted_context["speculative_inference"],
    "exploratory_speculative_count": len(exploratory_context["speculative_inference"]),
    "facet_relation_count": len(exploratory_context["facet_relations"]),
})

## What this notebook demonstrates

- speculative inference isolated in an `inference/*` branch
- manifold ranking metadata on inference candidates
- facet relation generation between related claims
- retrieval suppressing speculative results unless explicitly requested